# Clustering de estudiantes con responsabilidades parentales
**Prueba reproducible del análisis** — Universidad Nacional del Altiplano, Puno.

Este notebook reproduce el pipeline del artículo: limpieza, codificación,
estandarización, PCA, y tres algoritmos de clustering (K-Means, DBSCAN y
Agglomerative), con sus métricas de validación interna y las figuras.

> Sube el archivo `dataset_estudiantes_parentales.csv` cuando se te pida.

In [ ]:
# Colab ya trae pandas, numpy, scikit-learn y matplotlib.
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
np.random.seed(42)

## 1. Cargar el dataset

In [ ]:
# Opción A (Colab): subir el archivo
try:
    from google.colab import files
    up = files.upload()
    fname = list(up.keys())[0]
except Exception:
    fname = 'dataset_estudiantes_parentales.csv'  # Opción B: ruta local

df = pd.read_csv(fname, encoding='latin-1', sep=None, engine='python')
print('Registros:', len(df), '| Columnas:', df.shape[1])
df.head()

## 2. Limpieza y preparación
Se deriva la edad desde la fecha de nacimiento, se codifican las escalas
Likert (`Nunca`=0 … `Siempre`=3) y la carrera, y se eliminan registros
incompletos para quedar con la muestra analizada (~206).

In [ ]:
def calcular_edad(s):
    try:
        anio = int(str(s).split('/')[-1])
        return 2024 - anio if 1960 < anio < 2010 else np.nan
    except Exception:
        return np.nan

df['edad'] = df['f_nacimiento'].apply(calcular_edad)

likert = {'Nunca': 0, 'A veces': 1, 'Casi siempre': 2, 'Siempre': 3}

# Las 10 variables seleccionadas (chi-cuadrado) y su tipo
variables = {
    'carrera': 'cat',
    'cambio_dinamica_parental': 'likert',
    'comunicación_familiares': 'likert',
    'pausa_estudios_cuidado_hijo': 'likert',
    'considero_pausa_estudios': 'likert',
    'apoyo_familiar_hijo': 'likert',
    'apruebo_asignaturas': 'likert',
    'impacto_sobrecarga': 'likert',
    'impacto_rendimiento': 'likert',
    'edad': 'num',
}
nombres = {
    'carrera': 'Career', 'cambio_dinamica_parental': 'Parental dynamics change',
    'comunicación_familiares': 'Family communication',
    'pausa_estudios_cuidado_hijo': 'Study break for childcare',
    'considero_pausa_estudios': 'Considered study pause',
    'apoyo_familiar_hijo': 'Family childcare support',
    'apruebo_asignaturas': 'Subject pass rate',
    'impacto_sobrecarga': 'Overload impact',
    'impacto_rendimiento': 'Performance impact', 'edad': 'Age',
}

d = df[list(variables)].copy()
for c, t in variables.items():
    if t == 'likert':
        d[c] = d[c].map(likert)
    elif t == 'cat':
        d[c] = d[c].astype('category').cat.codes.replace(-1, np.nan)

d = d.dropna().reset_index(drop=True)
print('Registros tras limpieza:', len(d))
d.head()

## 3. Estandarización y PCA

In [ ]:
X = StandardScaler().fit_transform(d.values)

pca_full = PCA().fit(X)
cum = np.cumsum(pca_full.explained_variance_ratio_)
n95 = int(np.argmax(cum >= 0.95) + 1)
print(f'Componentes para >=95% de varianza: {n95} de {X.shape[1]}')

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(cum)+1), cum, 'o-')
plt.axhline(0.95, color='red', ls='--', lw=1)
plt.xlabel('Number of Principal Components'); plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance by PCA Components')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Proyección 2D solo para visualizar los clusters
X2 = PCA(n_components=2, random_state=42).fit_transform(X)

## 4. Número de clusters: codo y silueta

In [ ]:
inertias, sils, Ks = [], [], range(2, 11)
for k in Ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(list(Ks), inertias, 'o-'); ax[0].set_title('Elbow Method')
ax[0].set_xlabel('Number of Clusters'); ax[0].set_ylabel('Inertia'); ax[0].grid(alpha=0.3)
ax[1].plot(list(Ks), sils, 'o-'); ax[1].set_title('Silhouette Scores')
ax[1].set_xlabel('Number of Clusters'); ax[1].set_ylabel('Silhouette Score'); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('k con mayor silueta:', list(Ks)[int(np.argmax(sils))])

## 5. Clustering y métricas de validación
K-Means y Agglomerative con k=2; DBSCAN con eps=1.5, min_samples=5.
El ruido de DBSCAN se excluye al calcular las métricas.

In [ ]:
def metricas(X, labels):
    mask = labels != -1
    if len(set(labels[mask])) < 2:
        return (np.nan, np.nan, np.nan)
    return (round(silhouette_score(X[mask], labels[mask]), 4),
            round(davies_bouldin_score(X[mask], labels[mask]), 4),
            round(calinski_harabasz_score(X[mask], labels[mask]), 4))

km = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X)
ag = AgglomerativeClustering(n_clusters=2, linkage='ward').fit_predict(X)
db = DBSCAN(eps=1.5, min_samples=5).fit_predict(X)

tabla = pd.DataFrame(
    {'K-Means (k=2)': metricas(X, km),
     'DBSCAN': metricas(X, db),
     'Agglomerative (k=2)': metricas(X, ag)},
    index=['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz'])
print(tabla)
print('\nDBSCAN clusters encontrados:', sorted(set(db)))

> **Nota.** K-Means reproduce de cerca los valores del artículo
(Silhouette ≈ 0.16, Davies-Bouldin ≈ 2.1). DBSCAN es muy sensible a `eps`;
si solo encuentra ruido, ajusta `eps` entre 1.0 y 3.0 para hallar la
configuración de tres grupos reportada.

## 6. Visualización de los clusters en el espacio PCA

In [ ]:
for nombre, lab in [('K-Means', km), ('DBSCAN', db), ('Agglomerative', ag)]:
    plt.figure(figsize=(5.2, 4))
    sc = plt.scatter(X2[:, 0], X2[:, 1], c=lab, cmap='tab10', s=18)
    plt.title(f'{nombre} Clustering')
    plt.xlabel('PCA Component 1'); plt.ylabel('PCA Component 2')
    plt.tight_layout(); plt.show()

## 7. Perfiles de los clusters (K-Means)

In [ ]:
perf = d.copy(); perf['cluster'] = km
resumen = perf.groupby('cluster').mean().round(2)
resumen.columns = [nombres[c] for c in resumen.columns]
print(resumen.T)

---
### Notas finales
- La selección de variables (chi-cuadrado) requiere una variable objetivo;
  aquí se usan directamente las 10 variables ya reportadas en el artículo.
- DBSCAN y la semilla aleatoria pueden cambiar ligeramente las métricas.
- Pequeñas diferencias frente al artículo se deben a la codificación y al
  preprocesamiento exactos.